# 04 · Your first agent (the loop)

**Course:** Building LLM Agents, notebook 4 of 10
**Where we are:** in notebook 03 we did one manual tool round-trip by hand. The model asked for a tool, we ran it, we handed back the result, and the model wrote its answer. That single round-trip is the atom. Today we put it in a loop, and that loop is an agent.

### What you'll learn
- The one-sentence definition you'll keep coming back to: agent = LLM + tools + a loop
- The reason, act, observe cycle (ReAct in spirit) and how the model decides when it's done
- How to write `run_agent(task, tools, ...)`, the simplest possible agent loop, from scratch
- Why you always cap the loop with a `max_steps` guard
- How to watch a task that needs several tool calls solve itself, step by step
- How to package the loop into reusable `Agent` and `Tool` classes, the exact ones in `helpers.py`

### Prerequisites
- Notebooks 01 to 03. You're comfortable with chat completions, messages and roles, and the tool round-trip: tool schemas, `tool_calls`, executing a tool, and appending a `tool` message.

> Mental model: an agent is not a new kind of model. It is the same chat model you've used since notebook 01, wrapped in a small `while` loop you control. The model supplies the thinking; the loop supplies the persistence. The fancier topics later in this course, such as memory, multiple agents, and orchestration, all hang off this one loop.

## 1. Recap: the manual round-trip from notebook 03

We'll rebuild the round-trip from notebook 03 so we can stare at it. The model gets a single tool, a calculator, and we ask it one arithmetic question.

First, the usual setup. From notebook 02 onward we just import the ready client from `helpers.py`.

![A manual round trip wrapped in a loop becomes an agent](figures/04-roundtrip-to-loop.svg)

*Figure: the manual round trip from notebook 03, wrapped in a while-loop, is an agent. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

In [ ]:
import json
from helpers import client, MODEL, show_messages

Now the tool. As in notebook 03, a tool is just a plain Python function plus a JSON-Schema description that tells the model how to call it.

In [ ]:
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, e.g. '3*(4+5)'."""
    # The empty __builtins__ keeps eval from reaching anything dangerous.
    return str(eval(expression, {"__builtins__": {}}, {}))

# The schema the model sees — identical shape to notebook 03.
calc_schema = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression like '3*(4+5)'.",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"],
        },
    },
}

# A tiny registry so we can look a tool up by name when the model calls it.
TOOLS = {"calculator": calculator}
TOOL_SCHEMAS = [calc_schema]

Here is the round-trip, written out by hand exactly as in notebook 03: two model calls with our tool execution sandwiched in between.

In [ ]:
messages = [
    {"role": "system", "content": "You are careful. Use the calculator tool for ALL arithmetic."},
    {"role": "user", "content": "What is 3 * 149?"},
]

# --- Call 1: the model decides to use the tool ---------------------------
r1 = client.chat.completions.create(
    model=MODEL, messages=messages, tools=TOOL_SCHEMAS, temperature=0,
)
msg = r1.choices[0].message
messages.append(msg.model_dump(exclude_none=True))   # record the assistant turn

# --- Run the tool the model asked for ------------------------------------
for call in msg.tool_calls:
    args = json.loads(call.function.arguments or "{}")
    result = TOOLS[call.function.name](**args)
    print(f"tool: {call.function.name}({args}) -> {result}")
    messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})

# --- Call 2: the model writes its final answer using the result ----------
r2 = client.chat.completions.create(
    model=MODEL, messages=messages, tools=TOOL_SCHEMAS, temperature=0,
)
print("\nFINAL:", r2.choices[0].message.content)

That works, but look closely at the shape of it. Call the model, run a tool, call the model again. We hard-coded exactly two model calls because we happened to know one tool call would be enough.

> Key idea: what if the task needs the tool twice? Or five times? We'd have to write a third call, a fourth, and so on. We don't know in advance how many steps a task takes, so the model shouldn't either, and neither should our code. The fix is to stop hard-coding the number of round-trips and put the round-trip in a loop.

![The agent loop: reason, act, observe, with its exit condition](figures/04-agent-loop.svg)

*Figure: the agent loop, reason, act, observe, until the model stops asking for tools. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

## 2. The reason, act, observe cycle

Here's the pattern an agent repeats. It's the spirit of ReAct (*Reason + Act*):

1. **Reason:** the model looks at the conversation so far and decides what to do next.
2. **Act:** if it wants a tool, it emits a `tool_call` (that's `finish_reason == "tool_calls"`).
3. **Observe:** we run the tool and append the result as a `tool` message.
4. **Loop:** we call the model again with the new observation. Repeat.

The model exits the loop on its own. The moment it answers with plain text and no `tool_calls`, it's telling us "I have enough, here's the answer." That's the whole trick.

> Mental model: reason, act, observe, reason, act, observe, and so on until the answer. The model decides each turn whether to act again or to stop. Our loop just keeps offering it the chance and feeding back what happened.

## 3. The simplest possible agent: a `run_agent` function

Now we turn the recap into a loop. The function is small enough to read in one breath:

- send the messages and tools to the model;
- if the reply has no `tool_calls`, we're done, so return its text;
- otherwise run each tool it asked for, append the results, and go round again.

Two things to carry over from notebook 03, exactly:

- record the assistant turn with `msg.model_dump(exclude_none=True)`;
- record each tool result as `{"role": "tool", "tool_call_id": call.id, "content": str(result)}`.

> Gotcha, always cap the loop: a model can get stuck calling tools forever, or ping-pong between two of them. An uncapped `while True` is a runaway bill and a hung notebook. We add a `max_steps` guard so the loop cannot run away.

In [ ]:
def run_agent(task, tools, tool_schemas, system="", max_steps=8, verbose=True):
    """A minimal agent: an LLM in a loop until it stops asking for tools."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": task})

    for step in range(1, max_steps + 1):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tool_schemas, temperature=0,
        )
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))   # record assistant turn

        # No tool calls -> the model is done. Return its answer.
        if not msg.tool_calls:
            if verbose:
                print(f"[step {step}] done.")
            return msg.content, messages

        # Otherwise: ACT + OBSERVE — run each tool, feed the result back.
        for call in msg.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments or "{}")
            result = tools[name](**args) if name in tools else f"ERROR: no tool {name!r}"
            if verbose:
                print(f"[step {step}] {name}({args}) -> {result}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})

    # We fell out of the loop without a plain-text answer.
    return "[stopped: reached max_steps]", messages

That's it. That function is a complete agent, about fifteen lines of real code. We'll run it on the same easy question first, to confirm it behaves like the manual version: one tool call, then done.

In [ ]:
answer, _ = run_agent(
    task="What is 3 * 149?",
    tools=TOOLS,
    tool_schemas=TOOL_SCHEMAS,
    system="You are careful. Use the calculator tool for ALL arithmetic.",
)
print("\nFINAL:", answer)

## 4. A task that genuinely needs several steps

The single-step case is boring; the manual version handled it too. The loop earns its keep when a task needs the tool more than once, where each step depends on the previous result.

Try: "What is 15% of (3·149 + 2·25)?" To answer this the model has to compute the inner sum first, see the number, and only then take 15% of it. Watch it chain tool calls, reason, act, observe, repeat, and decide for itself when it's finished.

In [ ]:
answer, messages = run_agent(
    task="What is 15% of (3*149 + 2*25)?",
    tools=TOOLS,
    tool_schemas=TOOL_SCHEMAS,
    system="You are careful. Use the calculator tool for ALL arithmetic. "
           "Take one step at a time.",
)
print("\nFINAL:", answer)

Two true facts: `3*149 + 2*25 = 497`, and `15% of 497 = 74.55`. The model never did that arithmetic itself. It leaned on the tool at each step, and the loop is what let it take the second step after seeing the first result.

Now we replay the whole conversation with `show_messages` so the reason, act, observe rhythm is visible. Notice the alternation: the assistant asks for a tool, a `tool` message carries the result back, the assistant asks again, and the final assistant turn has plain content and no tool call. That's the exit.

In [ ]:
show_messages(messages)

> Why you care: you just watched the model decompose a problem on its own. It was never told "first compute the sum, then take the percentage." Given a loop and a tool, it planned the steps itself. That self-directed, multi-step behaviour is the reason we say "agent" instead of "chatbot".

> Cost: every trip round the loop resends the whole growing message list. A 4-step task is roughly 4 times the prompt tokens of a 1-step task, and they add up fast. `max_steps` caps the worst case; shorter histories (notebook 05) keep the typical case cheap.

## 5. Refactor: package the loop into `Agent` and `Tool` classes

The function works, but two things are clumsy in practice:

- We pass `tools` and `tool_schemas` around as two separate arguments that must stay in sync.
- There's no tidy place to keep an agent's name, system prompt, and settings together.

The standard fix is to wrap each function-plus-schema in a small `Tool` object, and wrap the loop in an `Agent` object that holds its own role and tools. We'll build them to match the constructor exactly, because we're about to reveal that these very classes already live in `helpers.py`, and notebooks 06 to 10 import them.

First, `Tool`: a function, its description, its JSON-Schema parameters, and a `.schema` property that emits the dict the API wants.

In [ ]:
class Tool:
    """Wraps a plain Python function so an LLM can call it."""

    def __init__(self, fn, description=None, parameters=None, name=None):
        self.fn = fn
        self.name = name or fn.__name__
        self.description = description or (fn.__doc__ or "").strip().split("\n")[0]
        self.parameters = parameters or {"type": "object", "properties": {}}

    @property
    def schema(self) -> dict:
        """The tool definition in the shape the API expects."""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.parameters,
            },
        }

    def __call__(self, **kwargs):
        return self.fn(**kwargs)

Now `Agent`: the same loop as `run_agent`, just with the messages bookkeeping split into a `run` (start from a task string) and a `run_messages` (run over an existing list, which we'll reuse for memory in notebook 05). The body is the reason, act, observe loop you already wrote, line for line.

In [ ]:
class Agent:
    """An LLM in a loop: call the model, run any tools it asks for, repeat."""

    def __init__(self, name="agent", system="", tools=None, model=None,
                 max_steps=8, temperature=0, client=None):
        self.name = name
        self.system = system
        self.tools = {t.name: t for t in (tools or [])}
        self.model = model or MODEL
        self.max_steps = max_steps
        self.temperature = temperature
        self.client = client or globals()["client"]

    def run(self, task, verbose=False):
        """Run the agent on a task string and return its final text answer."""
        messages = []
        if self.system:
            messages.append({"role": "system", "content": self.system})
        messages.append({"role": "user", "content": task})
        return self.run_messages(messages, verbose=verbose)

    def run_messages(self, messages, verbose=False):
        """Run the loop over an existing message list (lets you keep history)."""
        tool_schemas = [t.schema for t in self.tools.values()] or None
        for step in range(1, self.max_steps + 1):
            resp = self.client.chat.completions.create(
                model=self.model, messages=messages,
                tools=tool_schemas, temperature=self.temperature,
            )
            msg = resp.choices[0].message
            messages.append(msg.model_dump(exclude_none=True))   # verified append format

            if not msg.tool_calls:
                return msg.content              # no tools requested -> done

            for call in msg.tool_calls:
                name = call.function.name
                try:
                    args = json.loads(call.function.arguments or "{}")
                except json.JSONDecodeError:
                    args = {}
                if name in self.tools:
                    try:
                        result = self.tools[name](**args)
                    except Exception as e:      # feed errors back so the model can recover
                        result = f"ERROR running {name}: {e}"
                else:
                    result = f"ERROR: no tool named {name!r}"
                if verbose:
                    print(f"  [{self.name}] step {step}: {name}({args}) -> {result}")
                messages.append(
                    {"role": "tool", "tool_call_id": call.id, "content": str(result)}
                )

        return "[stopped: reached max_steps without a final answer]"

A couple of small but real upgrades over the bare function, all of which you'll want once tasks get messier:

- A `try/except` around tool execution, so a tool that throws no longer crashes the loop. The error text is fed back as the observation, giving the model a chance to recover (the bedrock of error handling in notebook 06).
- `run_messages` keeps the loop separate from where the history comes from, so we can hand it a saved conversation in notebook 05.

Now we run our `Agent` on the multi-step task. The `verbose=True` flag prints each step so the audience can watch the loop turn.

In [ ]:
calc = Tool(
    calculator,
    description="Evaluate a basic arithmetic expression like '3*(4+5)'.",
    parameters={"type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"]},
)

mathbot = Agent(
    name="mathbot",
    system="You are careful. Use the calculator tool for ALL arithmetic. Take one step at a time.",
    tools=[calc],
)

print(mathbot.run("What is 15% of (3*149 + 2*25)?", verbose=True))

## 6. We've already packaged this for you

That `Agent` and `Tool` you just wrote? They're the heart of this course, so we've put exactly those classes in `helpers.py`. From here on you don't redefine the loop, you just import it:

```python
from helpers import Agent, Tool
```

Nothing is hidden in there: open `helpers.py` and you'll find the same loop, line for line. Now we prove it behaves identically by re-running the same task through the imported version.

In [ ]:
from helpers import Agent as HelpersAgent, Tool as HelpersTool

calc2 = HelpersTool(
    calculator,
    description="Evaluate a basic arithmetic expression like '3*(4+5)'.",
    parameters={"type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"]},
)

mathbot2 = HelpersAgent(
    name="mathbot",
    system="You are careful. Use the calculator tool for ALL arithmetic. Take one step at a time.",
    tools=[calc2],
)

print(mathbot2.run("What is 15% of (3*149 + 2*25)?", verbose=True))

Same loop, same steps, same answer, because it is the same code. From notebook 06 onward, `from helpers import Agent, Tool` is all you'll need to spin up an agent.

## 7. What's still missing

You now have a working agent. But it's a minimal one, and a few obvious gaps are each their own notebook ahead:

- **Memory across runs.** Every call to `.run(task)` starts from a blank message list. The agent forgets everything the moment it returns. Giving it a memory that survives between turns (and fitting that into the context window) is notebook 05.
- **A real toolbox plus robustness.** One calculator is a toy. A useful agent has several tools, validates their arguments, and handles failures gracefully. Hardening a single agent is notebook 06.
- **More than one agent.** For bigger jobs you'll hand work between specialised agents (a researcher, a writer, a checker) that pass messages to each other. That's notebooks 07 to 09.

> Rule of thumb: reach for an agent loop only when a task needs more than one step or tool. If a single call answers it, a single call is cheaper, faster, and easier to debug. The loop is both power and expense, so spend it deliberately.

## Recap

You can now:
- state the definition cold: agent = LLM + tools + a loop
- describe the reason, act, observe cycle and how the model exits it (a reply with no `tool_calls`)
- write `run_agent(...)`, a complete agent loop, from scratch, reusing the verified append pattern
- always cap the loop with a `max_steps` guard so it can't run away
- run a task that requires multiple chained tool calls and watch the steps
- use the reusable `Agent` and `Tool` classes from `helpers.py`, the exact loop you built by hand

### Try it yourself
1. Drop `max_steps` to `1` on the multi-step task and re-run. What does the agent return, and why?
2. Add a second tool, say `def length(text): return len(text)` wrapped in a `Tool`, and ask a question that needs both tools. Watch which it calls first.
3. Make the calculator raise on a bad expression (for example, division by zero) and ask the agent to divide by zero. Confirm the `try/except` feeds the error back and the agent recovers or reports it.
4. Set `verbose=False` and instead pass the returned `messages` to `show_messages`. Compare the two ways of inspecting a run.

### Next up
05 · Memory & context. Right now the agent forgets everything between runs. We'll give it a memory: a conversation buffer, summarisation to stay small, and lexical retrieval to pull back the right past facts, all while respecting the context-window limit you first met in notebook 01.